# Battery cycling dashboard

Browse the battery cells and test procedures on a wiki, pick any
combination, and look at what was measured.

Run the cells below from top to bottom. The first two take a moment, then
the dashboard appears and you work in it:

- **Cells** and **Procedures** on the left: open a group and tick what you
  want to see. Ticking on both sides finds the measurements that match.
  Open one group at a time and let it load before opening the next.
- **Time Series** draws the signals over test time. **Axes & Units**
  changes what is on each axis, and in which unit.
- **Cycle Series** and **Batch Series** summarize capacity and coulomb
  efficiency per cycle, for one dataset or averaged over a batch.

In the wiki's own JupyterLite sandbox you are already signed in and nothing
is asked of you. Run it anywhere else and it needs a credentials file; see
the connection cell.

In [ ]:
# Kernel bootstrap, JupyterLite only: installs the OSW stack from the
# site's own wheel index in an order micropip cannot derive, and patches
# pyodide_http so requests reach the wiki at all. Safe to re-run. Absent
# on every other kernel, which the cells below handle.
try:
    import osw_bootstrap

    await osw_bootstrap.setup()
except ImportError:
    osw_bootstrap = None

In [ ]:
import importlib.util
import sys

if importlib.util.find_spec("opensemantic.batteries") is None:
    if sys.platform == "emscripten":
        # micropip resolves no extras, so what `[osl]` stands for is named
        # here. email-validator covers the pydantic[email] the models ask for:
        # without it the install succeeds and the import then fails.
        import piplite

        await piplite.install(
            [
                "opensemantic.batteries",
                "panel",
                "panelini",
                "email-validator",
            ],
            keep_going=True,
        )
    else:
        import subprocess

        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "opensemantic.batteries[osl]",
            ],
            check=True,
        )

## Connection

In the sandbox the browser's own session is used and there is nothing to
configure. Anywhere else, point `OSW_CRED_FILEPATH` at a `.pwd.yaml` with an
entry for the wiki, and `OSB_WIKI_DOMAIN` at the wiki itself.

In [ ]:
import os
from pathlib import Path

from osw.core import OSW

if osw_bootstrap is not None:
    # wtsite() binds to the wiki this notebook was served from and
    # authenticates with the browser's own cookie.
    osw_obj = OSW(site=osw_bootstrap.wtsite())
else:
    from opensemantic.batteries.view import connect_osw

    osw_obj = connect_osw(
        Path(os.environ.get("OSW_CRED_FILEPATH", "accounts.pwd.yaml")),
        wiki_domain=os.environ.get(
            "OSB_WIKI_DOMAIN", "wiki-dev.open-semantic-lab.org"
        ),
    )

osw_obj

## Dashboard

In [ ]:
import panel as pn

from opensemantic.batteries.v1 import CyclingDataRow
from opensemantic.batteries.view import LazyBatteryDataView, OSLBatteryBackend

pn.extension()

view = LazyBatteryDataView(
    backend=OSLBatteryBackend(
        osw_obj=osw_obj,
        row_class=CyclingDataRow,
        field_names=[
            "test_time",
            "voltage",
            "current",
            "capacity",
            "cycle_count",
            "step_type",
        ],
    ),
    title="Battery Cycling Dashboard",
    show_log=True,
)

# Wrapped in a fixed height on purpose. Panelini's .content is height:100%,
# which resolves against its parent - and a notebook output cell has no
# height of its own, so everything below the header collapses to nothing.
pn.Column(view.panel(), height=900, sizing_mode="stretch_width")

Nothing plotted? There may be no measurement for that combination of cell
and procedure. The **Log Console** at the bottom of the dashboard says what
was looked for.

The dashboard is 900 pixels tall. Change the `height=` in the cell above if
you want more or less room.